# 03 - Production: M3GNet

There are no methodological questions here: the parameters have already been
validated in notebook `00`. This notebook loops over the materials and writes
the JSON files to `data/results/m3gnet/`. It is intentionally short and repetitive.

**Separate environment** (`requirements/m3gnet.txt`): the three software stacks
cannot coexist in the same environment. `mace_mp(default_dtype="float64")`
calls `torch.set_default_dtype`, which is global state: instantiating CHGNet
after MACE in the same session causes the first linear layer to fail with
`expected mat1 and mat2 to have the same dtype`.
Restart the kernel between models.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.paths import RESULTS, REFERENCE, FIGURES
import src.materials as M
print("ROOT:", ROOT)
print("REFERENCE:", REFERENCE)
print("FIGURES:", FIGURES)

ROOT: c:\Users\antoj\Desktop\Tonio\projects\mlip-phonon-benchmark
REFERENCE: C:\Users\antoj\Desktop\Tonio\projects\mlip-phonon-benchmark\data\reference
FIGURES: C:\Users\antoj\Desktop\Tonio\projects\mlip-phonon-benchmark\figures


In [2]:
import logging
import warnings
import tensorflow as tf


# ------------------------------------------------------------
# Suppress known harmless warnings caused by running legacy
# M3GNet 0.2.4 with a modern TensorFlow / ASE stack.
# ------------------------------------------------------------

class M3GNetTensorFlowWarningFilter(logging.Filter):
    _ignored_messages = (
        "Detecting that an object or model or tf.train.Checkpoint "
        "is being deleted with unrestored values",

        "Value in checkpoint could not be found in the restored object",

        "You are casting an input of type complex64 "
        "to an incompatible dtype float32",

        "calling function (from tensorflow.python.eager."
        "polymorphic_function.polymorphic_function) "
        "with experimental_relax_shapes is deprecated",

        "The name tf.get_default_graph is deprecated",

        "The name tf.train.NewCheckpointReader is deprecated",
    )

    def filter(self, record):
        message = record.getMessage()
        return not any(
            ignored in message
            for ignored in self._ignored_messages
        )


tf.get_logger().addFilter(M3GNetTensorFlowWarningFilter())


# m3gnet 0.2.4 still uses atoms.set_calculator(calc).
# Modern ASE prefers atoms.calc = calc.
warnings.filterwarnings(
    "ignore",
    message=r"Please use atoms\.calc = calc",
    category=FutureWarning,
    module=r"m3gnet\.models\._dynamics",
)

In [3]:
import numpy as np
import ase
import ase.constraints
import tensorflow as tf

from importlib.metadata import version
from ase.filters import ExpCellFilter


# ============================================================
# Compatibility fix 1:
# m3gnet 0.2.4 expects ExpCellFilter in ase.constraints,
# while recent ASE versions moved it to ase.filters.
# ============================================================

if not hasattr(ase.constraints, "ExpCellFilter"):
    ase.constraints.ExpCellFilter = ExpCellFilter


from m3gnet.models import M3GNet, Potential, M3GNetCalculator
import m3gnet.graph._structure as m3_structure


# ============================================================
# Compatibility fix 2:
# legacy M3GNet creates PBC arrays with dtype=int.
# On Windows this is not necessarily int64, while modern
# pymatgen's Cython neighbor search requires np.int64.
# ============================================================

if not getattr(
    m3_structure.find_points_in_spheres,
    "_m3gnet_windows_int64_fix",
    False,
):
    _original_find_points_in_spheres = (
        m3_structure.find_points_in_spheres
    )

    def _find_points_in_spheres_compat(*args, **kwargs):
        if "pbc" in kwargs:
            kwargs["pbc"] = np.asarray(
                kwargs["pbc"],
                dtype=np.int64,
            )

        return _original_find_points_in_spheres(
            *args,
            **kwargs,
        )

    _find_points_in_spheres_compat._m3gnet_windows_int64_fix = True

    m3_structure.find_points_in_spheres = (
        _find_points_in_spheres_compat
    )


# ============================================================
# M3GNet model used by Loew et al.
# ============================================================

MODEL_NAME = "MP-2021.2.8-EFS"

tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)

model = M3GNet.load(MODEL_NAME)
potential = Potential(model=model)

calc = M3GNetCalculator(
    potential=potential,
    stress_weight=0.01,
)

MODEL_VERSION = (
    f"{MODEL_NAME}; "
    f"m3gnet={version('m3gnet')}; "
    f"tensorflow={tf.__version__}; "
    f"ase={ase.__version__}"
)

print("Model:", MODEL_NAME)
print("m3gnet:", version("m3gnet"))
print("TensorFlow:", tf.__version__)
print("ASE:", ase.__version__)

Model: MP-2021.2.8-EFS
m3gnet: 0.2.4
TensorFlow: 2.15.1
ASE: 3.29.0


In [4]:
from pymatgen.core import Lattice, Structure
from m3gnet.models import Relaxer

mo = Structure(
    Lattice.cubic(3.3),
    ["Mo", "Mo"],
    [
        [0.0, 0.0, 0.0],
        [0.5, 0.5, 0.5],
    ],
)

relaxer = Relaxer(
    potential=potential,
    optimizer="FIRE",
    relax_cell=True,
)

result = relaxer.relax(
    mo,
    verbose=False,
)

final_structure = result["final_structure"]

a_final = final_structure.lattice.abc[0]
energy_per_atom = (
    result["trajectory"].energies[-1] / len(mo)
)

print("a_final =", a_final, "Å")
print("E_final =", energy_per_atom, "eV/atom")

a_final = 3.1679769176880894 Å
E_final = -10.859207153320312 eV/atom


In [5]:
from src.reference import load_reference
from src.phonons import ph_to_ase

mp_id = M.MATERIALS["Si"]["mp_id"]

ph_ref = load_reference(
    mp_id,
    functional="pbe",
    use_nac=False,
)

test_atoms = ph_to_ase(ph_ref.unitcell)
test_atoms.calc = calc

print("Energy [eV]:")
print(test_atoms.get_potential_energy())

print("\nMax |force| [eV/A]:")
forces = test_atoms.get_forces()
print((forces**2).sum(axis=1).max()**0.5)

print("\nStress [eV/A^3]:")
print(test_atoms.get_stress())

Energy [eV]:
-42.86964

Max |force| [eV/A]:
2.476654096754323e-07

Stress [eV/A^3]:
[ 5.07954182e-03  5.07954182e-03  5.07954462e-03  4.63402650e-09
 -1.50379287e-09  6.04717831e-10]


In [6]:
import time, json
from src.phonons import compute_phonons, thermal_properties, band_structure, make_band_path
from src.io_utils import make_payload, save_result
from src.reference import load_reference
from src.phonons import ph_to_ase

MODEL = "m3gnet"
DTYPE = "float32"
FMAX = 0.005
DISP = 0.01

materials = list(M.MATERIALS)      # or M.QUICK for a quick test
print("materials:", materials)

for key in materials:
    out = RESULTS / MODEL / f"{key}.json"
    if out.exists():
        print(f"[skip] {key} already computed")
        continue
    try:
        t0 = time.perf_counter()
        mp_id = M.MATERIALS[key]["mp_id"]
        
        ph_ref = load_reference(
            mp_id,
            functional="pbe",
            use_nac=False,
        )

        band_path = make_band_path(
            ph_ref,
            npoints=101,
        )
        
        atoms = ph_to_ase(ph_ref.unitcell)
        ph, relaxed = compute_phonons(
            atoms,
            calc,
            supercell_matrix=ph_ref.supercell_matrix,
            primitive_matrix=ph_ref.primitive_matrix,
            disp=DISP,
            fmax=FMAX,
            fix_symmetry=True,
            logfile=None,
        )
        tp = thermal_properties(ph)
        bands = band_structure(ph, band_path)
        payload = make_payload(key, MODEL, MODEL_VERSION, DTYPE, ph, relaxed,
                               DISP, FMAX, tp, bands,
                               runtime_s=round(time.perf_counter() - t0, 2))
        save_result(MODEL, key, payload)
        print(f"[ok] {key:5s} omega_max={payload['omega_max_THz']:7.3f} THz  "
              f"imag={payload['has_imaginary']}  ({payload['runtime_s']}s)")
    except Exception as e:
        print(f"[FAIL] {key}: {type(e).__name__}: {e}")

materials: ['Si', 'SiC', 'AlP', 'ZnS', 'MgO', 'NaCl', 'AlN', 'GaN']
[ok] Si    omega_max=  8.469 THz  imag=False  (1.85s)
[ok] SiC   omega_max= 19.925 THz  imag=False  (1.34s)
[ok] AlP   omega_max=  8.966 THz  imag=False  (0.99s)
[ok] ZnS   omega_max=  8.184 THz  imag=False  (1.49s)
[ok] MgO   omega_max= 16.113 THz  imag=False  (1.59s)
[ok] NaCl  omega_max=  5.549 THz  imag=False  (1.49s)
[ok] AlN   omega_max= 22.706 THz  imag=False  (4.14s)
[ok] GaN   omega_max= 18.322 THz  imag=False  (4.06s)
